# RENA Ground-Truth Unlearning Testbed (Colab / Local GPU Edition)

This notebook runs empirical unlearning experiments using `Qwen2.5-0.5B-Instruct`.
It implants synthetic canary knowledge into a LoRA adapter, verifies baseline recall (>80%), executes real unlearning procedures (GA, GradDiff, NPO, suppression controls), and performs multi-angle behavioral audits with an independent oracle.

In [ ]:
# 1. Environment & Hardware Verification
!pip install -q transformers peft accelerate scipy pydantic pyyaml
import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Executing on device: {device}")

In [ ]:
# 2. Clone RENA repository or import testbed modules
import sys
from pathlib import Path
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from core.testbed_real.dataset import FORGET_FACTS, RETAIN_FACTS
from core.testbed_real.implant import implant_knowledge, evaluate_recall
from core.testbed_real.unlearn import get_unlearned_model
from core.testbed_real.oracle import compute_target_log_likelihood, run_relearning_attack

print(f"Loaded {len(FORGET_FACTS)} forget facts and {len(RETAIN_FACTS)} retain facts.")

In [ ]:
# 3. Execute Knowledge Implantation with LoRA
implant_knowledge('configs/testbed/smoke.yaml', 'checkpoints/implanted_lora')

In [ ]:
# 4. Evaluate Unlearning Methods & Behavioral Audits
methods = ['reference', 'retained', 'suppress_refusal', 'suppress_prompt', 'ga', 'graddiff', 'npo']
results = {}

for m in methods:
    model, tok = get_unlearned_model(m, 'Qwen/Qwen2.5-0.5B-Instruct', 'checkpoints/implanted_lora', device)
    ev_f = evaluate_recall(model, tok, FORGET_FACTS, device)
    ev_r = evaluate_recall(model, tok, RETAIN_FACTS, device)
    ll = compute_target_log_likelihood(model, tok, FORGET_FACTS, device)
    results[m] = {
        'forget_recall': ev_f['recall'],
        'retain_utility': ev_r['recall'],
        'mean_target_log_prob': ll['mean_log_prob'],
    }
    print(f"Method {m:18s} | Forget Recall: {ev_f['recall']*100:5.1f}% | Retained Utility: {ev_r['recall']*100:5.1f}% | LogProb: {ll['mean_log_prob']:.2f}")

In [ ]:
# 5. Independent Oracle: Relearning Attack Benchmark
print("Running relearning vulnerability attacks...")
for m in ['suppress_refusal', 'ga', 'graddiff']:
    model, tok = get_unlearned_model(m, 'Qwen/Qwen2.5-0.5B-Instruct', 'checkpoints/implanted_lora', device)
    attack = run_relearning_attack(model, tok, device, steps=8)
    print(f"Method {m:18s} -> Steps to recovery: {attack['steps_to_recovery']} | Assessment: {attack['vulnerability_assessment']}")